In [1]:
import random
import torch
import os
import umap
import time
import glob
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.decomposition import PCA, FastICA, FactorAnalysis
from sklearn.random_projection import GaussianRandomProjection
from sklearn.manifold import TSNE, trustworthiness
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.simplefilter("ignore", UserWarning)

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
# Helper function for KNN evaluation. 
def knn_preservation_score(X_high, X_low, k=5):
    knn_high = NearestNeighbors(n_neighbors=k).fit(X_high)
    knn_low = NearestNeighbors(n_neighbors=k).fit(X_low)
    neighbors_high = knn_high.kneighbors(return_distance=False)
    neighbors_low = knn_low.kneighbors(return_distance=False)
    score = np.mean([
        len(set(high).intersection(set(low))) / k
        for high, low in zip(neighbors_high, neighbors_low)
    ])
    return score


# Helper function for continuity evaluation. 
# Formula adapted from https://link.springer.com/chapter/10.1007/3-540-44668-0_68#preview and https://www.sciencedirect.com/science/article/pii/S0893608006000724 (M2(k) formula)
def continuity(X_high, X_low, k=5):
    # Number of samples.
    n = X_high.shape[0]
    # Build rank matrices.
    nn_high = NearestNeighbors(n_neighbors=n-1).fit(X_high)
    rank_matrix_high = nn_high.kneighbors(return_distance=False)
    nn_low = NearestNeighbors(n_neighbors=k + 1).fit(X_low)
    neighbors_low = nn_low.kneighbors(return_distance=False)[:,1:]
    
    penalty = 0
    for i in range(n):
        ranks = {point: rank for rank, point in enumerate(rank_matrix_high[i], start=1)}
        
        for j in neighbors_low[i]:
            if j not in rank_matrix_high[i][:k]:  # j is not a high-d neighbor
                penalty += ranks.get(j, n) - k

    denom = n * k * (2 * n - 3 * k - 1)
    continuity_score = 1 - (2 / denom) * penalty
    return continuity_score


# Helper function for all dimensionality reduction methods. 
def all_dim_red_methods(inputs_df):
    # Do not need standard scaler as inputs are already z-score normalized. 
    inputs = inputs_df[[f'factor_{i}' for i in range(1, 7)]].values
    methods = {
        "PCA": PCA(n_components=2, random_state=RANDOM_STATE),
        "ICA": FastICA(n_components=2, random_state=RANDOM_STATE),
        "RP": GaussianRandomProjection(n_components=2, random_state=RANDOM_STATE),
        "t-SNE": TSNE(n_components=2, random_state=RANDOM_STATE),
        # 10 neighbors works well in most scenarios - https://umap-learn.readthedocs.io/en/latest/how_umap_works.html
        "UMAP": umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=10),
        "FA": FactorAnalysis(n_components=2, random_state=RANDOM_STATE)
    }

    results = []
    for name, model in methods.items():
        start = time.time()
        trans_embeddings = model.fit_transform(inputs)
        end = time.time()
        results.append({
            "Method": name,
            # using k = 5 as it is common (https://www.nature.com/articles/s42003-022-03628-x)
            "Trustworthiness": trustworthiness(inputs, trans_embeddings, n_neighbors=5),
            "KNN Preservation": knn_preservation_score(inputs, trans_embeddings, k=5),
            "Continuity": continuity(inputs, trans_embeddings, k=5),
            "Time": end-start
        })
    return pd.DataFrame(results)

# Helper function for plotting dimensionality reduction results. 
def plot_results(df_results, save=False):
    # make df to work with
    df_normalized = pd.DataFrame()
    df_normalized['Method'] = df_results['Method']
    df_normalized['Normalized Trustworthiness'] = (df_results['Trustworthiness'] - df_results['Trustworthiness'].min()) / (df_results['Trustworthiness'].max() - df_results['Trustworthiness'].min())   
    df_normalized['Normalized KNN Preservation'] = (df_results['KNN Preservation'] - df_results['KNN Preservation'].min()) / (df_results['KNN Preservation'].max() - df_results['KNN Preservation'].min())   
    df_normalized['Normalized Continuity'] = (df_results['Continuity'] - df_results['Continuity'].min()) / (df_results['Continuity'].max() - df_results['Continuity'].min())   
    df_normalized['Normalized Time'] = (df_results['Time'] - df_results['Time'].min()) / (df_results['Time'].max() - df_results['Time'].min())   


    sns.set_theme(style='whitegrid')
    plt.figure(figsize=(16, 12))

    titles = ['Trustworthiness', 'KNN Preservation', 'Continuity', 'Time']

    for i, title in enumerate(titles, start=1):
        plt.subplot(2, 2, i)
        sns.barplot(data=df_results, x="Method", y=title, hue="Method", palette="Set2", dodge=False, legend=False)
        plt.title(title)
        plt.ylim(0, df_results[title].max()+0.1)
        plt.ylabel("Score (0-1)")
        plt.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

    # save normalized metrics figure
    plt.figure(figsize=(6, 4))

    df_melted = df_normalized.melt(id_vars="Method", var_name="Metric", value_name="Normalized Score")
    sns.barplot(data=df_melted, x="Method", y="Normalized Score", hue="Metric", palette="Set2")
    plt.title("Normalized Metrics")
    plt.ylim(0, 1.05)
    plt.ylabel("Score (0-1)")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    if save:
        # save plot
        if os.path.exists(f'../figures/dimRed/'):
            plt.savefig("../figures/dimRed/normalizedMetricsAllModels.png", bbox_inches='tight', dpi=300, format='png')
        else:
            os.makedirs(f'../figures/dimRed/', exist_ok=False)
            plt.savefig("../figures/dimRed/normalizedMetricsAllModels.png", bbox_inches='tight', dpi=300, format='png')

    plt.show()

# Helper function to make embeddings from chosen dimensionality reduction algorithm.
def put_embeddings_in_df(input_df, method):
    model = method(n_components=2, random_state=RANDOM_STATE)

    trans_embeddings = model.fit_transform(input_df[[f'factor_{i}' for i in range(1, 7)]].values)

    input_df['x'] = [trans_embedding[0] for trans_embedding in trans_embeddings]
    input_df['y'] = [trans_embedding[1] for trans_embedding in trans_embeddings]
    return input_df

# Helper function for drawing dimensionality reduction plot coloured by category. 
def draw_dim_red_plot(df):

    fig = px.scatter(
        df,
        x='x',
        y='y',
        color='category',
        hover_data=['doc_id'],
        opacity=0.8,
        width=800,
        height=600,
        title=""
    )

    # Optionally make markers smaller
    fig.update_traces(marker=dict(size=8, line=dict(color='black', width=1)))  # outline each marker

    fig.show()


In [5]:
def dimensionality_reduction_plots(file_paths="./outputsTrain/*/mean_model_zero_shot_classification.csv"):    
    # Get all CSV files.
    all_files = glob.glob(file_paths)  # change this to your folder path

    # Loop through files.
    all_dfs = []

    for f in all_files:
        df = pd.read_csv(f)
        # Extract categories from doc_id
        doc_ids = df['doc_id']
        df['category'] = df['doc_id'].apply(lambda x: x.split('_')[0])
        
        # Aggregate numeric factor columns by category
        factor_cols = [f"factor_{i}" for i in range(1, 7)]
        df_agg = df.groupby('category')[factor_cols].mean().reset_index()
        df_agg['doc_id'] = doc_ids
        all_dfs.append(df_agg)

    # Concatenate all aggregated DataFrames
    dfs = pd.concat(all_dfs, ignore_index=True)

    # dim_red_results = all_dim_red_methods(dfs)
    # plot_results(dim_red_results)
    # Use UMAP as chosen method.
    dfs = put_embeddings_in_df(dfs, umap.UMAP)
    draw_dim_red_plot(dfs)

In [6]:
dimensionality_reduction_plots(file_paths="./outputsTrain/*/mean_model_zero_shot_classification.csv")

In [7]:
dimensionality_reduction_plots(file_paths="./outputsTest/*/mean_model_zero_shot_classification.csv")

In [8]:
dimensionality_reduction_plots(file_paths="./outputsAll/*/mean_model_zero_shot_classification.csv")

In [9]:
dimensionality_reduction_plots(file_paths="./outputsTrain/*/mda_dim_scores.csv")

In [10]:
dimensionality_reduction_plots(file_paths="./outputsTest/*/mda_dim_scores.csv")

In [11]:
dimensionality_reduction_plots(file_paths="./outputsAll/*/mda_dim_scores.csv")